# Phase 1: Evaluate with Built-in Tasks

This notebook demonstrates the simplest way to run Korean LLM evaluations using TrustyAI's **built-in task support** via `taskList.taskNames`.

## How it works

TrustyAI ships with a curated set of lm-evaluation-harness tasks (Tier 1 & Tier 2). You simply specify the task name and the operator handles everything.

## Available Korean Tasks

| Task Name | Dataset | Tier | Description |
|-----------|---------|------|-------------|
| `kmmlu_direct_law` | KMMLU | 1 | Korean MMLU - Law subject (direct) |
| `kmmlu_hard_law` | KMMLU-Hard | 2 | Korean MMLU Hard - Law subject |
| `kobest_wic` | KoBEST | 2 | Word-in-Context disambiguation |
| `haerae_history` | HAE-RAE | 2 | Korean history knowledge |
| `hrm8k_ksm` | HRM8K | 2 | Korean School Math reasoning |

## Step 1: Configuration

Set your environment-specific values:

In [ ]:
NAMESPACE = "hyo-project"
MODEL_NAME = "vllm-gemma4-e2b"                    # InferenceService name = served_model_name
TOKENIZER = "google/gemma-2b"
BASE_URL = f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1/completions"
TASK_NAME = "kmmlu_direct_law"                    # Change this to run different tasks
LIMIT = 5                                          # Number of samples (remove for full eval)
JOB_NAME = f"eval-{TASK_NAME.replace('_', '-')}"

print(f"Model: {MODEL_NAME}")
print(f"Task: {TASK_NAME}")
print(f"Base URL: {BASE_URL}")
print(f"Job Name: {JOB_NAME}")

## Step 2: Review the YAML Template

Let's look at the LMEvalJob YAML that will be applied:

In [ ]:
!cat samples/eval-builtin.yaml

## Step 3: Generate Customized YAML

We'll substitute our configuration values into the template:

In [ ]:
yaml_content = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: LMEvalJob
metadata:
  name: {JOB_NAME}
spec:
  allowOnline: true
  model: local-completions
  modelArgs:
    - name: base_url
      value: "{BASE_URL}"
    - name: model
      value: "{MODEL_NAME}"
    - name: tokenizer
      value: "{TOKENIZER}"
    - name: verify_certificate
      value: "False"
    - name: num_concurrent
      value: "1"
    - name: max_retries
      value: "3"
    - name: tokenized_requests
      value: "false"
  taskList:
    taskNames:
      - {TASK_NAME}
  logSamples: true
  limit: "{LIMIT}"
  pod:
    container:
      env:
        - name: HF_TOKEN
          valueFrom:
            secretKeyRef:
              name: hf-token
              key: HF_TOKEN
        - name: OPENAI_API_KEY
          valueFrom:
            secretKeyRef:
              name: lmeval-sa-token
              key: token
"""

with open("/tmp/eval-job.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

## Step 4: Submit the Evaluation Job

In [ ]:
!oc apply -f /tmp/eval-job.yaml -n {NAMESPACE}

## Step 5: Monitor Execution

Wait for the Pod to complete (typically 30-60 seconds for a small `limit`):

In [ ]:
import time

for i in range(12):
    !oc get pods -n {NAMESPACE} | grep {JOB_NAME}
    time.sleep(10)

## Step 6: Get Results

In [ ]:
!oc logs {JOB_NAME} -c main -n {NAMESPACE} 2>&1 | grep -A 5 "Tasks\|Metric"

### Full JSON Results

In [ ]:
import json
import subprocess

result = subprocess.run(
    ["oc", "get", "lmevaljob", JOB_NAME, "-n", NAMESPACE,
     "-o", "jsonpath={.status.results}"],
    capture_output=True, text=True
)

if result.stdout:
    results = json.loads(result.stdout)
    print(json.dumps(results.get("results", {}), indent=2))
else:
    print("Results not yet available. Check pod status.")

## Step 7: Cleanup

In [ ]:
!oc delete lmevaljob {JOB_NAME} -n {NAMESPACE}

## Summary

**When to use Phase 1 (Built-in Tasks):**

- Quick smoke tests to verify model serving works
- CI/CD pipelines where you need a fast evaluation gate
- When the built-in Korean tasks cover your needs
- Minimal YAML configuration

**Limitations:**

- Only TrustyAI Tier 1/2 tasks are available
- Cannot run the full KMMLU (45 subjects) or CLIcK (11 categories) benchmark
- No access to community-contributed tasks

**Next:** See `2_custom_tasks/1_custom_task_eval.ipynb` for full flexibility with Git-sourced tasks.